In [1]:
import pickle
import regex as re

In [2]:
merges = pickle.load(open("../TinyStoriesV2-GPT4-train.txt.merges.pkl", 'rb'))
vocab = pickle.load(open("../TinyStoriesV2-GPT4-train.txt.vocab.pkl", 'rb'))

In [6]:
vocab[9999]

b'<|endoftext|>'

In [5]:
len(merges)

9743

In [6]:
len(vocab)

10000

In [9]:
max(vocab, key=lambda o: len(vocab[o]))

7162

In [10]:
vocab[7162]

b' accomplishment'

`merges` -> list of tuples -> tuples having byte(s) pairs

sentence -> pre-tokenize -> words

words -> break into bytes -> merge each pair in the order of their occurrence in `merges`

In [90]:
x = " this word is bussing"

In [128]:
xp = tuple([bytes([o]) for o in list(" this".encode("utf-8"))])
xp

(b' ', b't', b'h', b'i', b's')

In [181]:
xp = tuple([bytes([o]) for o in list("honestly".encode("utf-8"))])

i = 0
while len(xp) > 1 and i+1 < len(xp):
    # print(f"checking: {(xp[i], xp[i+1])}; i: {i}; xp[{i}]: {xp[i]}")
    for p in merges:
        if (xp[i], xp[i+1]) == p:
            # print("found: ", i)
            xp = (*xp[:i], xp[i]+xp[i+1], *xp[i+2:])
            i = 0 # start searching from the beginning again
            break
    else: # if pair is not found in merges
        i += 1

xp

(b'ho', b'ne', b'st', b'ly')

In [182]:
rvocab = {v:k for k,v in vocab.items()}

In [184]:
rvocab[b'ho']

6940

In [ ]:
def encode_subword(subword: str):
    xp = tuple([bytes([o]) for o in list(subword.encode("utf-8"))])

    i = 0
    while len(xp) > 1 and i+1 < len(xp):
        for p in merges:
            if (xp[i], xp[i+1]) == p:
                xp = (*xp[:i], xp[i]+xp[i+1], *xp[i+2:])
                i = 0 # start searching from the beginning again
                break
        else: # if pair is not found in merges
            i += 1

    return [rvocab[o] for o in xp]

In [176]:
encode_subword("honestly")

[6940, 6832, 350, 460]

In [9]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [194]:
subwords = []
for subword in re.finditer(PAT, x, re.IGNORECASE):
    subwords.append(subword.captures()[0])

subwords

[' this', ' word', ' is', ' bussing']

In [196]:
for o in subwords:
    print(encode_subword(o))

[726]
[1034, 114, 100]
[430]
[861, 559, 298]


In [ ]:
def encode(text: str):
    subwords = []
    for subword in re.finditer(PAT, text, re.IGNORECASE):
        subwords.append(subword.captures()[0])

    token_ids = []
    for subword in subwords:
        token_ids.extend(encode_subword(subword))
    
    return token_ids

encode(" this shit's bussin'")

[726, 1034, 114, 100, 430, 861, 559, 298]

In [204]:
t = encode(" this shit's bussin'")
t

[726, 1034, 114, 100, 430, 861, 559, 298]

In [208]:
for o in t:
    print(vocab[o])

b' this'
b' wo'
b'r'
b'd'
b' is'
b' bu'
b'ss'
b'ing'


In [3]:
text1 = "hi this is a test<|endoftext|>this is another sentence"
text2 = "hi this is a test. there is not another sentence ... wait"

In [17]:
escaped_tokens = [re.escape(o) for o in ["<|endoftext|>"]]
special_tok_pat = '|'.join(escaped_tokens)
special_tok_pat

'<\\|endoftext\\|>'

In [22]:
re.split(special_tok_pat, text1)

['hi this is a test', 'this is another sentence']

In [28]:
PAT + '|' + special_tok_pat

"'(?:[sdmt]|ll|ve|re)| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+|<\\|endoftext\\|>"

In [31]:
subwords = []
for part in re.split(special_tok_pat, text1):
    for subword in re.finditer(PAT, part, re.IGNORECASE):
        subwords.append(subword.captures()[0])

In [30]:
subwords

['hi', ' this', ' is', ' a', ' test', 'this', ' is', ' another', ' sentence']

In [36]:
for part in re.finditer(special_tok_pat, text1):
    print(part)

<regex.Match object; span=(17, 30), match='<|endoftext|>'>


In [40]:
text1[30]

't'

In [3]:
from tokenizer import Tokenizer

In [4]:
tok = Tokenizer(
    vocab=vocab,
    merges=merges,
    special_tokens=["<|endoftext|>"]
)

In [5]:
text1 = "hi this is a test<|endoftext|>this is another sentence"

In [6]:
text1

'hi this is a test<|endoftext|>this is another sentence'

In [7]:
out = ""
for id in tok.encode(text1):
    out += bytes.decode(vocab[id], errors='replace')

In [8]:
out

'hi this is a test<|endoftext|>this is another sentence'

In [9]:
text1

'hi this is a test<|endoftext|>this is another sentence'

In [14]:
text1 = "hi this is a test<|endoftext|>this is another sentence<|endoftext|>"

In [15]:
out = ""
for id in tok.encode(text1):
    out += bytes.decode(vocab[id], errors='replace')

out

'hi this is a test<|endoftext|>this is another sentence<|endoftext|>'

In [16]:
text1.split("<|endoftext|>")

['hi this is a test', 'this is another sentence', '']